# Snippet from Cookbook.md


In [ ]:
"""Read a saved certificate JSON (from `compitum route --trace > file.json`)
and print a short text analysis. There is no src/compitum/plotting.py module
and no bundled PNG-generating script in this repo -- this sticks to plain
text; if you want charts, pip install matplotlib and plot utility_components
yourself."""
from __future__ import annotations

import argparse
import json
from pathlib import Path


def main() -> int:
    ap = argparse.ArgumentParser(description="Summarize a saved certificate JSON.")
    ap.add_argument("cert_path", type=Path)
    args = ap.parse_args()

    data = json.loads(args.cert_path.read_text())

    print(f"Model: {data['model']}")
    print(f"Utility: {data['utility']:.4f}")
    print("Components:")
    for k, v in sorted(data["utility_components"].items(), key=lambda kv: abs(kv[1]), reverse=True):
        print(f"  {k}: {v:+.4f}")

    b = data["boundary"]
    print(f"Boundary: winner={b['winner']} runner_up={b['runner_up']} "
          f"gap={b['utility_gap']:.4f} entropy={b['entropy']:.4f} is_boundary={b['is_boundary']}")

    c = data["constraints"]
    print(f"Feasible: {c['feasible']} (status={c['status']})")
    shadow = c.get("shadow_prices", {})
    active = {k: v for k, v in shadow.items() if abs(v) > 1e-9}
    if active:
        print("Active shadow prices:")
        for k, v in sorted(active.items(), key=lambda kv: abs(kv[1]), reverse=True):
            print(f"  {k}: {v:.4f}")
    else:
        print("No binding constraints (all shadow prices are 0).")

    d = data["drift"]
    print(f"Trust radius: {d['trust_radius']:.4f}  Drift EMA: {d['drift_ema']:.4f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
